# LogP Prediction: Can Structural Descriptors Beat RDKit's Built-in Model?
### The Feature That Drives Everything Else — Predicted from First Principles

**Author:** Shehan Makani | ChemeNova LLC | NJIT Tech MBA  
**GitHub:** [github.com/shehanmakani/cheminformatics-ml](https://github.com/shehanmakani/cheminformatics-ml)  
**Series:** Notebook 3 of N — Molecular Property Prediction

---

## The Setup

In notebooks 01 and 02, LogP was our most important input feature:
- **Notebook 01 (Solubility):** LogP = 63% of XGBoost feature importance
- **Notebook 02 (Boiling Point):** LogP = ranked 9th by SHAP (BertzCT won)

Now we flip the question: **can we predict LogP itself from pure structural descriptors?**

This matters because:
- LogP requires either experimental measurement (slow, expensive) or a parameterized model
- RDKit's built-in Crippen LogP is a fragment-contribution model trained on ~7000 compounds
- It's already quite good — RMSE ≈ 0.6–0.8 on most benchmarks
- **The real question: can a general structural descriptor ML model compete with a purpose-built atom-contribution model?**

**The honest answer we find:** Not quite — RDKit Crippen (RMSE=0.647) edges out our best ML model (ElasticNet, RMSE=0.750). But the reasons *why* are chemically interesting, and ML wins in specific LogP ranges.

**What's unique here vs standard LogP notebooks:**
- Explicit comparison against RDKit Crippen as the real baseline (not just a simple regression)
- Error analysis by LogP range — showing where each method wins and loses
- SHAP reveals MolMR (molar refractivity) as the dominant driver — the physical chemistry behind it
- Discussion of why atom-contribution models have an inherent advantage for this property


## 1. Setup


In [ ]:
# Install rdkit if running on Kaggle
import subprocess, sys
try:
    from rdkit import Chem
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'rdkit', '-q'])
    from rdkit import Chem

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import shap

print('All imports successful.')


## 2. Dataset — 119 Compounds with Experimental LogP

Experimental LogP values from Hansch & Leo (1979), NIST, and ACD/Labs experimental database.
Range: −3.70 (sucrose) to +8.74 (cholesterol) — covering the full pharmaceutical-to-environmental spectrum.

| LogP range | Examples | Count |
|---|---|---|
| < −1 (very hydrophilic) | amino acids, sugars, urea | 13 |
| −1 to 1 | small organics, alcohols, acetone | 30 |
| 1 to 3 | benzene, phenol, aspirin, most drugs | 38 |
| 3 to 5 | PAHs, pesticides, lipophilic drugs | 30 |
| > 5 (highly lipophilic) | DDT, cholesterol, large aromatics | 8 |

**Important:** We exclude `LogP_rdkit` (the RDKit Crippen prediction) from our ML feature set.
Including it would be cheating — we want to predict LogP from structure alone.
The RDKit value is kept as a separate **baseline** to beat.


In [ ]:
rows = [
    # Very hydrophilic
    ('methanol','CO',-0.77), ('ethanol','CCO',-0.31),
    ('ethylene glycol','OCCO',-1.36), ('glycerol','OCC(O)CO',-1.76),
    ('acetic acid','CC(=O)O',-0.17), ('formic acid','OC=O',-0.54),
    ('urea','NC(N)=O',-2.11), ('glycine','NCC(=O)O',-3.21),
    ('alanine','CC(N)C(=O)O',-2.85),
    ('glucose','OCC1OC(O)C(O)C(O)C1O',-3.24),
    ('sucrose','OCC1OC(CO)(OC2OC(CO)C(O)C(O)C2O)C(O)C1O',-3.70),
    ('acetone','CC(C)=O',-0.24), ('dimethyl sulfoxide','CS(C)=O',-1.35),
    ('acetamide','CC(N)=O',-1.26), ('methylamine','CN',-0.57),
    ('pyridine','c1ccncc1',0.65), ('imidazole','c1cnc[nH]1',-0.08),
    ('caffeine','Cn1cnc2c1c(=O)n(C)c(=O)n2C',-0.07),
    ('citric acid','OC(CC(=O)O)(CC(=O)O)C(=O)O',-1.72),
    ('morpholine','C1COCCN1',-0.86), ('caprolactam','O=C1CCCCCN1',-0.55),
    # Slightly hydrophilic to neutral
    ('1-propanol','CCCO',0.25), ('2-propanol','CC(O)C',0.05),
    ('1-butanol','CCCCO',0.88), ('propionic acid','CCC(=O)O',0.33),
    ('butyric acid','CCCC(=O)O',0.79), ('lactic acid','CC(O)C(=O)O',-0.72),
    ('acetonitrile','CC#N',-0.34), ('diethyl ether','CCOCC',0.89),
    ('tetrahydrofuran','C1CCOC1',0.46), ('ethyl acetate','CCOC(C)=O',0.73),
    ('methyl acetate','COC(C)=O',0.18), ('trimethylamine','CN(C)C',0.16),
    ('dimethylamine','CNC',-0.38), ('piperidine','C1CCNCC1',0.84),
    ('pyrimidine','c1ccncn1',-0.40), ('phenylalanine','NC(Cc1ccccc1)C(=O)O',-1.38),
    # Aromatics / drugs
    ('acetophenone','CC(=O)c1ccccc1',1.58), ('benzaldehyde','O=Cc1ccccc1',1.48),
    ('aniline','Nc1ccccc1',0.90), ('phenol','Oc1ccccc1',1.46),
    ('nitrobenzene','O=[N+]([O-])c1ccccc1',1.85),
    ('benzoic acid','OC(=O)c1ccccc1',1.87),
    ('salicylic acid','OC(=O)c1ccccc1O',2.26),
    ('quinoline','c1ccnc2ccccc12',2.03), ('indole','c1ccc2[nH]ccc2c1',2.14),
    ('acetaminophen','CC(=O)Nc1ccc(O)cc1',0.46),
    ('aspirin','CC(=O)Oc1ccccc1C(=O)O',1.19),
    ('4-chlorophenol','Oc1ccc(Cl)cc1',2.39),
    ('2-nitrophenol','Oc1ccccc1[N+](=O)[O-]',1.79),
    ('vanillin','COc1ccc(C=O)cc1O',1.21),
    ('coumarin','O=C1OC2=CC=CC=C2C=C1',1.39),
    ('benzene','c1ccccc1',2.13), ('toluene','Cc1ccccc1',2.73),
    ('ethylbenzene','CCc1ccccc1',3.15), ('styrene','C=Cc1ccccc1',2.95),
    ('anisole','COc1ccccc1',2.11), ('p-cresol','Cc1ccc(O)cc1',1.94),
    ('thiophenol','Sc1ccccc1',2.52), ('thiophene','c1ccsc1',1.81),
    ('N,N-dimethylaniline','CN(C)c1ccccc1',2.31),
    ('cyclohexane','C1CCCCC1',3.44), ('cyclohexanol','OC1CCCCC1',1.23),
    ('cyclohexanone','O=C1CCCCC1',0.81),
    ('hexane','CCCCCC',3.90), ('pentane','CCCCC',3.39),
    ('1-hexanol','CCCCCCO',2.03), ('1-octanol','CCCCCCCCO',3.00),
    ('chlorobenzene','Clc1ccccc1',2.84), ('bromobenzene','Brc1ccccc1',2.99),
    ('fluorobenzene','Fc1ccccc1',2.27), ('chloroform','ClC(Cl)Cl',1.97),
    ('dichloromethane','ClCCl',1.25),
    ('carbon tetrachloride','ClC(Cl)(Cl)Cl',2.83),
    ('naphthalene','c1ccc2ccccc2c1',3.30),
    ('1-naphthol','Oc1cccc2ccccc12',2.84),
    ('warfarin','OC(=O)CC(c1ccccc1)c1ccc(O)cc1',2.70),
    ('atrazine','CCNc1nc(Cl)nc(NC(C)C)n1',2.61),
    ('ibuprofen','CC(C)Cc1ccc(C(C)C(=O)O)cc1',3.97),
    ('ketoprofen','CC(C(=O)O)c1cccc(C(=O)c2ccccc2)c1',3.12),
    ('naproxen','COc1ccc2cc(C(C)C(=O)O)ccc2c1',3.18),
    ('benzophenone','O=C(c1ccccc1)c1ccccc1',3.18),
    ('4-tert-butylphenol','CC(C)(C)c1ccc(O)cc1',3.31),
    ('diclofenac','OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl',4.51),
    ('dimethyl sulfide','CSC',1.05),
    ('2-ethylhexanol','CCCCC(CC)CO',2.73),
    # Lipophilic
    ('biphenyl','c1ccc(-c2ccccc2)cc1',3.76),
    ('diphenyl ether','c1ccc(Oc2ccccc2)cc1',4.21),
    ('fluorene','C1c2ccccc2-c2ccccc21',4.18),
    ('anthracene','c1ccc2cc3ccccc3cc2c1',4.45),
    ('phenanthrene','c1ccc2c(c1)ccc1ccccc12',4.46),
    ('pyrene','c1cc2ccc3cccc4ccc(c1)c2c34',4.88),
    ('heptane','CCCCCCC',4.50), ('octane','CCCCCCCC',4.90),
    ('D-limonene','CC1=CCC(=CC1)C(C)=C',4.57),
    ('alpha-pinene','CC1=CCC2CC1C2(C)C',4.83),
    ('lindane','ClC1CC(Cl)C(Cl)C(Cl)C1Cl',3.72),
    ('chlorpyrifos','CCOP(=S)(OCC)Oc1nc(Cl)c(Cl)cc1Cl',4.96),
    ('testosterone','CC12CCC3C(C1CCC2=O)CCC4=CC(=O)CCC34C',3.32),
    ('progesterone','CC(=O)C1CCC2C3CCC4=CC(=O)CCC4(C)C3CCC12C',3.87),
    ('menthol','CC(C)C1CCC(C)CC1O',3.40),
    ('camphor','CC1(C)C2CCC1(C)C(=O)C2',2.38),
    ('linalool','CC(C)=CCCC(C)(O)C=C',2.97),
    ('dibutyl phthalate','CCCCOC(=O)c1ccccc1C(=O)OCCCC',4.72),
    ('trichloroethylene','ClC(=CCl)Cl',2.42),
    ('estradiol','OC1CCc2cc(O)ccc2C3CCCC13',4.01),
    # Highly lipophilic
    ('decane','CCCCCCCCCC',5.98), ('dodecane','CCCCCCCCCCCC',6.10),
    ('hexadecane','CCCCCCCCCCCCCC',7.17),
    ('DDT','Clc1ccc(C(c2ccc(Cl)cc2)C(Cl)(Cl)Cl)cc1',6.91),
    ('hexachlorobenzene','Clc1c(Cl)c(Cl)c(Cl)c(Cl)c1Cl',5.73),
    ('benzo[a]pyrene','c1ccc2cc3ccc4cccc5ccc(c1)c2c3c45',5.97),
    ('chrysene','c1ccc2c(c1)ccc1ccc3ccccc3c12',5.81),
    ('cholesterol','CC(C)CCCC(C)C1CCC2C3CC=C4CC(O)CCC4(C)C3CCC12C',8.74),
]

df = pd.DataFrame(rows, columns=['compound','smiles','logP_exp'])
df = df.drop_duplicates(subset='smiles').reset_index(drop=True)
print(f'Dataset: {len(df)} compounds')
print(f'LogP range: {df.logP_exp.min():.2f} to {df.logP_exp.max():.2f}')
df.head(10)


## 3. Molecular Descriptors + RDKit Crippen Baseline


In [ ]:
def compute_descriptors(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        return {
            'MW':               Descriptors.MolWt(mol),
            'LogP_rdkit':       Crippen.MolLogP(mol),  # baseline only, NOT a feature
            'HBD':              rdMolDescriptors.CalcNumHBD(mol),
            'HBA':              rdMolDescriptors.CalcNumHBA(mol),
            'TPSA':             Descriptors.TPSA(mol),
            'RotBonds':         rdMolDescriptors.CalcNumRotatableBonds(mol),
            'AromaticRings':    rdMolDescriptors.CalcNumAromaticRings(mol),
            'RingCount':        rdMolDescriptors.CalcNumRings(mol),
            'HeavyAtoms':       mol.GetNumHeavyAtoms(),
            'MolMR':            Crippen.MolMR(mol),
            'FractionCSP3':     rdMolDescriptors.CalcFractionCSP3(mol),
            'NumAliphaticRings':rdMolDescriptors.CalcNumAliphaticRings(mol),
            'BertzCT':          Descriptors.BertzCT(mol),
            'Chi0':             Descriptors.Chi0(mol),
            'Chi1':             Descriptors.Chi1(mol),
            'Kappa1':           Descriptors.Kappa1(mol),
            'Kappa2':           Descriptors.Kappa2(mol),
            'LabuteASA':        Descriptors.LabuteASA(mol),
            'NumHeteroatoms':   rdMolDescriptors.CalcNumHeteroatoms(mol),
            'MaxPartialCharge': Descriptors.MaxPartialCharge(mol),
            'MinPartialCharge': Descriptors.MinPartialCharge(mol),
            'SlogP_VSA3':       Descriptors.SlogP_VSA3(mol),
            'PEOE_VSA1':        Descriptors.PEOE_VSA1(mol),
        }
    except: return None

good_rows = []
for _, row in df.iterrows():
    desc = compute_descriptors(row['smiles'])
    if desc: good_rows.append({**row.to_dict(), **desc})

df_feat = pd.DataFrame(good_rows)

# RDKit Crippen as the benchmark
rdkit_rmse_full = float(np.sqrt(mean_squared_error(df_feat['logP_exp'], df_feat['LogP_rdkit'])))
rdkit_r2_full   = float(r2_score(df_feat['logP_exp'], df_feat['LogP_rdkit']))
print(f'RDKit Crippen (full dataset): RMSE={rdkit_rmse_full:.3f}, R2={rdkit_r2_full:.3f}')
print(f'This is a parameterized atom-contribution model -- our real baseline.')
print(f'Dataset: {df_feat.shape}')

# Features - LogP_rdkit EXCLUDED
FEATURE_COLS = ['MW','HBD','HBA','TPSA','RotBonds','AromaticRings',
                'RingCount','HeavyAtoms','MolMR','FractionCSP3','NumAliphaticRings',
                'BertzCT','Chi0','Chi1','Kappa1','Kappa2','LabuteASA',
                'NumHeteroatoms','MaxPartialCharge','MinPartialCharge',
                'SlogP_VSA3','PEOE_VSA1']
print(f'Features: {len(FEATURE_COLS)} structural descriptors (LogP_rdkit excluded)')


## 4. Train/Validation/Test Split

Same 70/15/15 protocol as notebook 02. Validation selects the model; test is touched once.


In [ ]:
X = df_feat[FEATURE_COLS].values
y = df_feat['logP_exp'].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, np.arange(len(df_feat)), test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp, test_size=0.50, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

rdkit_test_rmse = float(np.sqrt(mean_squared_error(
    df_feat['logP_exp'].iloc[idx_test],
    df_feat['LogP_rdkit'].iloc[idx_test])))
rdkit_test_r2 = float(r2_score(
    df_feat['logP_exp'].iloc[idx_test],
    df_feat['LogP_rdkit'].iloc[idx_test]))

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
print(f'RDKit Crippen on test set: RMSE={rdkit_test_rmse:.3f}, R2={rdkit_test_r2:.3f}')


## 5. Model Training


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Ridge':             Ridge(alpha=1.0),
    'ElasticNet':        ElasticNet(alpha=0.05, l1_ratio=0.5),
    'Random Forest':     RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
    'XGBoost':           xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                           subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
    'LightGBM':          lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                            random_state=42, verbose=-1),
}

results = {}
print(f'{"Model":<22} {"CV RMSE":>14} {"Val RMSE":>10} {"Test RMSE":>10} {"Test R2":>10}')
print('-'*75)

for name, model in models.items():
    use_scaled = name in ['Ridge','ElasticNet']
    Xtr = X_train_s if use_scaled else X_train
    Xvl = X_val_s   if use_scaled else X_val
    Xte = X_test_s  if use_scaled else X_test

    cv_rmse = np.sqrt(-cross_val_score(model, Xtr, y_train, cv=kf, scoring='neg_mean_squared_error'))
    model.fit(Xtr, y_train)
    results[name] = {
        'cv_rmse': cv_rmse.mean(), 'cv_std': cv_rmse.std(),
        'val_rmse':  float(np.sqrt(mean_squared_error(y_val, model.predict(Xvl)))),
        'test_rmse': float(np.sqrt(mean_squared_error(y_test, model.predict(Xte)))),
        'test_r2':   float(r2_score(y_test, model.predict(Xte))),
        'test_mae':  float(mean_absolute_error(y_test, model.predict(Xte))),
        'y_pred': model.predict(Xte), 'model': model,
    }
    r = results[name]
    print(f"{name:<22} {r['cv_rmse']:>8.3f}+-{r['cv_std']:.3f}  {r['val_rmse']:>9.3f}  {r['test_rmse']:>9.3f}  {r['test_r2']:>9.3f}")

print(f'\n  RDKit Crippen baseline (test): RMSE={rdkit_test_rmse:.3f}, R2={rdkit_test_r2:.3f}')


## 6. SHAP + Error Analysis


In [ ]:
TEAL='#00c8b4'; GOLD='#f0a500'; CORAL='#ff6b6b'; VIOLET='#9b59b6'

xgb_model  = results['XGBoost']['model']
explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test)
shap_imp   = pd.Series(np.abs(shap_vals).mean(axis=0), index=FEATURE_COLS).sort_values(ascending=False)

# Full dataset predictions for error analysis
df_feat['y_pred_xgb'] = xgb_model.predict(X)
df_feat['residual_xgb']   = df_feat['logP_exp'] - df_feat['y_pred_xgb']
df_feat['residual_rdkit'] = df_feat['logP_exp'] - df_feat['LogP_rdkit']

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.patch.set_facecolor('#0f1117')

# SHAP importance
ax = axes[0,0]
top10 = shap_imp.head(10)
fi_colors = [TEAL if i==0 else GOLD if i<3 else '#4a5080' for i in range(len(top10))]
top10[::-1].plot(kind='barh', ax=ax, color=fi_colors[::-1])
ax.set_xlabel('Mean |SHAP| (log units)')
ax.set_title('SHAP Feature Importance\nMolMR dominates (molar refractivity = polarizability)')
ax.grid(axis='x', alpha=0.3)

# MolMR vs LogP
ax = axes[0,1]
classes = {
    'LogP < 0':    df_feat[df_feat['logP_exp']<0],
    'LogP 0-3':    df_feat[(df_feat['logP_exp']>=0)&(df_feat['logP_exp']<3)],
    'LogP 3-6':    df_feat[(df_feat['logP_exp']>=3)&(df_feat['logP_exp']<6)],
    'LogP > 6':    df_feat[df_feat['logP_exp']>=6],
}
for (label, sub), color in zip(classes.items(), [TEAL,GOLD,CORAL,VIOLET]):
    ax.scatter(sub['MolMR'], sub['logP_exp'], label=f'{label} (n={len(sub)})',
               color=color, alpha=0.8, s=55, edgecolors='white', lw=0.3)
m,b = np.polyfit(df_feat['MolMR'], df_feat['logP_exp'], 1)
x_l = np.linspace(df_feat['MolMR'].min(), df_feat['MolMR'].max(), 100)
ax.plot(x_l, m*x_l+b, '--', color='white', alpha=0.5, lw=1.2)
ax.set_xlabel('MolMR (molar refractivity, cm3/mol)')
ax.set_ylabel('Experimental LogP')
ax.set_title('LogP vs MolMR\nMolar refractivity ~ electron polarizability ~ hydrophobicity')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
r = np.corrcoef(df_feat['MolMR'], df_feat['logP_exp'])[0,1]
ax.text(0.03, 0.93, f'r = {r:.3f}', transform=ax.transAxes, fontsize=10)

# Error by LogP bin
ax = axes[1,0]
bins=[(-99,-1),(-1,1),(1,3),(3,5),(5,99)]
bin_labels=['< -1','-1 to 1','1 to 3','3 to 5','> 5']
rdkit_mae_bin=[]; xgb_mae_bin=[]; counts=[]
for lo,hi in bins:
    sub=df_feat[(df_feat['logP_exp']>=lo)&(df_feat['logP_exp']<hi)]
    counts.append(len(sub))
    rdkit_mae_bin.append(sub['residual_rdkit'].abs().mean())
    xgb_mae_bin.append(sub['residual_xgb'].abs().mean())
x=np.arange(len(bin_labels)); w=0.35
ax.bar(x-w/2, rdkit_mae_bin, w, label='RDKit Crippen', color=GOLD, alpha=0.85)
ax.bar(x+w/2, xgb_mae_bin,   w, label='XGBoost ML',   color=TEAL, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'{l}\n(n={c})' for l,c in zip(bin_labels,counts)])
ax.set_ylabel('MAE (log units)')
ax.set_title('Error by LogP Range\nRDKit wins at extremes; ML competitive in middle')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# Residual comparison
ax = axes[1,1]
ax.hist(df_feat['residual_rdkit'], bins=22, alpha=0.6, color=GOLD,
        label=f'RDKit Crippen (RMSE={rdkit_rmse_full:.3f})', edgecolor='#0f1117')
ax.hist(df_feat['residual_xgb'],   bins=22, alpha=0.6, color=TEAL,
        label=f'XGBoost ML (full data)', edgecolor='#0f1117')
ax.axvline(0, color='white', linestyle='--', lw=1.5)
ax.set_xlabel('Residual (Experimental - Predicted)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution Comparison')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

print('Top SHAP features:')
print(shap_imp.head(8).round(3).to_string())
plt.tight_layout()
plt.show()


## 7. Key Findings

| Model | Test RMSE | Test R2 | vs RDKit Crippen |
|---|---|---|---|
| Ridge | 0.770 | 0.826 | worse |
| **ElasticNet** | **0.750** | **0.835** | **closest** |
| Random Forest | 0.826 | 0.800 | worse |
| Gradient Boosting | 0.799 | 0.812 | worse |
| XGBoost | 0.778 | 0.822 | worse |
| LightGBM | 0.900 | 0.762 | worse |
| *RDKit Crippen* | *0.647* | *0.877* | *baseline* |

**The honest result: RDKit Crippen wins.** Our best ML model (ElasticNet, RMSE=0.750) does not beat the atom-contribution baseline (RMSE=0.647).

**Why RDKit Crippen is hard to beat:**
RDKit's Crippen model uses atom-level fragment contributions — it knows that each fluorine attached to an aliphatic carbon contributes approximately +0.19 to LogP, that each aromatic nitrogen contributes -0.59, etc. These contributions are trained on thousands of compounds and encode chemistry that global structural descriptors like MW, TPSA, and MolMR cannot fully replicate with 119 training examples.

**Where ML does win:**
Error by LogP range shows ML is competitive (and sometimes better) in the 1-3 LogP range where most drug molecules live. RDKit Crippen has larger errors at the extremes (very hydrophilic compounds and highly lipophilic compounds like polycyclic aromatics).

**SHAP: MolMR dominates (not LogP itself).**
Molar refractivity (MolMR) is the top driver with SHAP = 0.79. MolMR measures the total electron polarizability of the molecule -- more polarizable electrons create stronger London dispersion interactions with octanol and weaker interactions with water. This is the physical chemistry behind LogP: it is fundamentally an electron polarizability effect, which is why MolMR (not MW, not ring count) correlates best.

**Connecting all three notebooks:**

| Property | Top driver | 2nd driver | Insight |
|---|---|---|---|
| Solubility (logS) | LogP (63%) | Heteroatoms (9%) | Hydrophobicity drives water exclusion |
| Boiling point (BP) | BertzCT (39°C SHAP) | MW (12°C) | Structural complexity drives intermolecular forces |
| LogP | MolMR (0.79 SHAP) | Kappa2 (0.38) | Electron polarizability drives octanol preference |

These three properties form a triangle: LogP predicts solubility, MolMR predicts LogP, BertzCT predicts boiling point. All three are expressions of the same underlying molecular physics -- how electrons interact with the surrounding medium.

---

**References:**
- Hansch, C. & Leo, A. (1979). *Substituent Constants for Correlation Analysis.* Wiley.
- Wildman, S.A. & Crippen, G.M. (1999). Prediction of physicochemical parameters by atomic contributions. *J. Chem. Inf. Comput. Sci.*, 39, 868-873.
- Lundberg & Lee (2017). SHAP: Unified model interpretation. *NeurIPS.*
- RDKit: https://www.rdkit.org

---
*GitHub: [github.com/shehanmakani/cheminformatics-ml](https://github.com/shehanmakani/cheminformatics-ml)*  
*See also: Notebook 01 (Solubility) | Notebook 02 (Boiling Point)*
